In [353]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sklearn.metrics import r2_score
import os 
import glob
import math
import scipy.signal
import pwlf


In [435]:
class Test:
    def __init__(self, df):
        self.df = df.copy()  # Avoid modifying the original DataFrame
        self.id = df['participant_ID'].iloc[0]
        self.date = df['visit_date'].iloc[0]
        self.name = f"{self.id}_{self.date}"
        
        # Assign 'Stage' only for 'EXERCISE' phase, then fill NaN values with 0
        self.df['Stage'] = np.nan
        exercise_mask = self.df['Phase'] == 'EXERCISE'
        self.df.loc[exercise_mask, 'Stage'] = self.df[exercise_mask].groupby(['Speed', 'Grade']).ngroup() + 1
        self.df['Stage']= self.df['Stage'].fillna(0)  # Ensure non-exercise rows have Stage 0
        
        # Convert 't' to timedelta and set it as index
        self.df['t'] = pd.to_timedelta(self.df['t'])
        self.df.set_index('t', inplace=True)
        
        self.lactate_df = self.prepare_lactate_df()
    
    def prepare_lactate_df(self):
        # Reset index to retrieve 't' column
        lactate_df = self.df.dropna(subset=['La-']).reset_index()
        lactate_df = lactate_df[['t', 'La-', 'Phase', 'Speed', 'Grade', 'Stage']]
        
        # Remove extreme lactate values (greater than 20)
        lactate_df = lactate_df[lactate_df['La-'] <= 20]
        
        # Apply logarithmic transformation, handling non-positive values
        lactate_df['log_lactate'] = np.log(lactate_df['La-'].replace(0, np.nan))
        lactate_df['log_speed'] = np.log(lactate_df['Speed'].replace(0, np.nan))
        
        return lactate_df

    def lt_ref_vals(self):
        lt1ref_index = (self.lactate_df['La-'].diff() >= 0.5)
        lt2ref_index = (self.lactate_df['La-'].diff() >= 1)
        LT1ref = self.lactate_df.loc[lt1ref_index.idxmax(), 'La-'] if lt1ref_index.any() else None
        LT2ref = self.lactate_df.loc[lt2ref_index.idxmax(), 'La-'] if lt2ref_index.any() else None
        LT1ref_speed = self.lactate_df.loc[lt1ref_index.idxmax(), 'Speed'] if lt1ref_index.any() else None
        LT2ref_speed = self.lactate_df.loc[lt2ref_index.idxmax(), 'Speed'] if lt2ref_index.any() else None
        results = {"LT1_ref":LT1ref, "LT2_ref":LT2ref, "LT1ref_speed":LT1ref_speed, "LT2ref_speed":LT2ref_speed}
        return results

    def lt_abs_vals(self):
        lt1abs_index = (self.lactate_df['La-'] >= 2)
        lt2abs_index = (self.lactate_df['La-'] >= 4)
        
        LT1abs = self.lactate_df.loc[lt1abs_index.idxmax(), 'La-'] if lt1abs_index.any() else None
        LT2abs = self.lactate_df.loc[lt2abs_index.idxmax(), 'La-'] if lt2abs_index.any() else None
        LT1abs_speed = self.lactate_df.loc[lt1abs_index.idxmax(), 'Speed'] if lt1abs_index.any() else None
        LT2abs_speed = self.lactate_df.loc[lt2abs_index.idxmax(), 'Speed'] if lt2abs_index.any() else None
        
        results = {
            "LT1_abs": LT1abs, 
            "LT2_abs": LT2abs, 
            "LT1abs_speed": LT1abs_speed, 
            "LT2abs_speed": LT2abs_speed
        }
        return results

    def v_slope_method(self, x_list, y_list):
        """Finds the optimal intersection point using the V-Slope method."""
        split_index = len(x_list) // 2
        best_index = split_index
        min_error = float('inf')
        
        for i in range(10, len(x_list) - 10):  # Ensure enough points for both regressions
            slope1, intercept1, _, _, _ = linregress(x_list[:i], y_list[:i])
            slope2, intercept2, _, _, _ = linregress(x_list[i:], y_list[i:])
            
            if slope1 < 1 and slope2 >= 1:  # Condition for V-slope method
                error = np.sum((y_list[:i] - (slope1 * x_list[:i] + intercept1))**2) + \
                        np.sum((y_list[i:] - (slope2 * x_list[i:] + intercept2))**2)
                if error < min_error:
                    min_error = error
                    best_index = i
        
        slope1, intercept1, _, _, _ = linregress(x_list[:best_index], y_list[:best_index])
        slope2, intercept2, _, _, _ = linregress(x_list[best_index:], y_list[best_index:])
        
        x_threshold = (intercept2 - intercept1) / (slope1 - slope2)
        y_threshold = slope1 * x_threshold + intercept1
        
        return best_index, slope1, intercept1, slope2, intercept2, x_threshold, y_threshold  
            
    def lt_log_semilog(self, plot=False):
        
    # === Prepare Data ===
        filtered_df = self.lactate_df.drop_duplicates(subset='Speed', keep='first')
        filtered_df = filtered_df[(filtered_df['Speed'] > 0) & (filtered_df['La-'] > 0)]

        log_speed = np.log(filtered_df['Speed'].values)
        log_lactate = np.log(filtered_df['La-'].values)
        speed = filtered_df['Speed'].values

        # === LT1: log(speed) vs log(lactate) ===
        my_pwlf1 = pwlf.PiecewiseLinFit(log_speed, log_lactate)
        breaks1 = my_pwlf1.fit(2)  # 2 segments
        x_intersect_lt1 = breaks1[1]
        y_intersect_lt1 = my_pwlf1.predict([x_intersect_lt1])[0]
        lt1_log = math.exp(y_intersect_lt1)
        lt1_log_speed = math.exp(x_intersect_lt1)
        lt1_log_r2 = my_pwlf1.r_squared()

        # === LT2: speed vs log(lactate) ===
        my_pwlf2 = pwlf.PiecewiseLinFit(speed, log_lactate)
        breaks2 = my_pwlf2.fit(2)
        x_intersect_lt2 = breaks2[1]
        y_intersect_lt2 = my_pwlf2.predict([x_intersect_lt2])[0]
        lt2_semilog = x_intersect_lt2
        lt2_semilog_speed = math.exp(y_intersect_lt2)
        lt2_semi_log_r2 = my_pwlf2.r_squared()

        # === Plotting ===
        if plot:
            fig, axs = plt.subplots(1, 2, figsize=(12, 5))

            # LT1 plot
            axs[0].scatter(log_speed, log_lactate, color='gray', label='Data')
            x_hat1 = np.linspace(min(log_speed), max(log_speed), 100)
            y_hat1 = my_pwlf1.predict(x_hat1)
            axs[0].plot(x_hat1, y_hat1, color='blue', label='pwlf fit')
            axs[0].scatter(x_intersect_lt1, y_intersect_lt1, color='black', zorder=3, label='LT1')
            axs[0].set_title('LT1: Log Speed vs Log Lactate')
            axs[0].set_xlabel('Log Speed')
            axs[0].set_ylabel('Log Lactate')
            axs[0].legend()
            axs[0].grid()

            # LT2 plot
            axs[1].scatter(speed, log_lactate, color='gray', label='Data')
            x_hat2 = np.linspace(min(speed), max(speed), 100)
            y_hat2 = my_pwlf2.predict(x_hat2)
            axs[1].plot(x_hat2, y_hat2, color='red', label='pwlf fit')
            axs[1].scatter(x_intersect_lt2, y_intersect_lt2, color='black', zorder=3, label='LT2')
            axs[1].set_title('LT2: Speed vs Log Lactate')
            axs[1].set_xlabel('Speed')
            axs[1].set_ylabel('Log Lactate')
            axs[1].legend()
            axs[1].grid()

            plt.tight_layout()
            plt.show()

        return {
            "LT1_log":  lt1_log,
            "LT1_log_speed": lt1_log_speed,
            "LT1_log_r2": lt1_log_r2,
            "LT2_semilog": lt2_semilog, 
            "LT2_semilog_speed": lt2_semilog_speed,
            "LT2_semi_log_r2": lt2_semi_log_r2
        }

    def vt1_vt2_vo2_vco2(self, plot=False):
        vo2 = self.df['VO2']
        vco2 = self.df['VCO2']
        ve = self.df['VE']
        
        # Define an initial split index to separate the two regression regions
        split_index = len(vo2) // 2
        
        # Function to find the optimal intersection point
        best_index = split_index
        min_error = float('inf')
        
        for i in range(10, len(vo2) - 10):  # Ensure enough points for both regressions
            slope1, intercept1, _, _, _ = linregress(vo2[:i], vco2[:i])
            slope2, intercept2, _, _, _ = linregress(vo2[i:], vco2[i:])
            
            if slope1 < 1 and slope2 >= 1:  # Condition for V-slope method
                error = np.sum((vco2[:i] - (slope1 * vo2[:i] + intercept1))**2) + \
                        np.sum((vco2[i:] - (slope2 * vo2[i:] + intercept2))**2)
                if error < min_error:
                    min_error = error
                    best_index = i
        
        # Compute best fit lines
        slope1, intercept1, _, _, _ = linregress(vo2[:best_index], vco2[:best_index])
        slope2, intercept2, _, _, _ = linregress(vo2[best_index:], vco2[best_index:])
        
        # Intersection point
        vt1_vo2_threshold = (intercept2 - intercept1) / (slope1 - slope2)
        vt1_vco2_threshold = slope1 * vt1_vo2_threshold + intercept1

        # Fit piecewise model with 2 segments (1 breakpoint)
        my_pwlf = pwlf.PiecewiseLinFit(vco2, ve)
        breaks = my_pwlf.fit(2)  # This returns x-values of breakpoints

        # VT2 is the breakpoint between segment 1 and 2
        vt2_vco2 = breaks[1]
        vt2_ve = my_pwlf.predict([vt2_vco2])[0]
        # Create plot
        if plot:
            fig, axs = plt.subplots(1, 2, figsize=(14, 6))
            # --- VT1 Plot ---
            axs[0].scatter(vo2, vco2, label='Data', color='lightgray')
            axs[0].plot(vo2[:best_index], slope1 * vo2[:best_index] + intercept1, 'b', label='Slope < 1')
            axs[0].plot(vo2[best_index:], slope2 * vo2[best_index:] + intercept2, 'r', label='Slope ≥ 1')
            axs[0].scatter(vt1_vo2_threshold, vt1_vco2_threshold, color='black', zorder=3, label='VT1')
            axs[0].set_xlabel('VO₂ (L/min)')
            axs[0].set_ylabel('VCO₂ (L/min)')
            axs[0].set_title('V-Slope Method (VT1)')
            axs[0].legend()
            axs[0].grid()

               # --- VT2 Plot using PWLF ---
            axs[1].scatter(vco2, ve, label='Data', color='lightgray')

            # Plot the fitted segments
            x_hat = np.linspace(min(vco2), max(vco2), 100)
            y_hat = my_pwlf.predict(x_hat)
            axs[1].plot(x_hat, y_hat, 'b-', label='PWLF Fit')

            # Mark VT2
            axs[1].scatter(vt2_vco2, vt2_ve, color='black', marker='^', zorder=3, label='VT2 (PWLF)')
            axs[1].axvline(vt2_vco2, color='black', linestyle='--', alpha=0.5)

            axs[1].set_xlabel('VCO₂ (L/min)')
            axs[1].set_ylabel('VE (L/min)')
            axs[1].set_title('Ventilatory Compensation Point (VT2)')
            axs[1].legend()
            axs[1].grid()
                
        return {"vt1_vo2_level_vo2_vo2":vt1_vo2_threshold, "vt1_vo2_level_vo2_vco2":vt1_vco2_threshold,
                'vt2_vco2': vt2_vco2, 'vt2_ve': vt2_vco2}
    
    def vt1_vt2_VE(self, window_size, plot=False):
        VE_VO2 = self.df['VE/VO2'].rolling(f'{window_size}s', min_periods=1).mean()
        VE_CO2 = self.df['VE/VCO2'].rolling(f'{window_size}s', min_periods=1).mean()

        # Convert time index to total seconds
        time = self.df.index.total_seconds()
        # Identify nadir: Minimum VE/VO2 value
        nadir_index = VE_VO2.idxmin()
        if pd.isna(nadir_index) or nadir_index not in self.df.index:
            raise ValueError("nadir_index is NaN or not in DataFrame index")

        nadir_time = self.df.index[self.df.index.get_loc(nadir_index)]  # Safer method to get position
        nadir_value = VE_VO2[nadir_index]

        # Find the first rise after nadir while VE/CO2 is constant or increasing
        rise_index = None

        nadir_seconds = nadir_time.total_seconds()  # Convert Timedelta to seconds
        end_seconds = self.df.index[-1].total_seconds()
        for index in VE_CO2.index[1:]:
            prev_index = VE_CO2.index[VE_CO2.index < index].max()
            if VE_CO2.loc[index] >= VE_CO2.loc[prev_index] and VE_VO2.loc[index] > nadir_value:
                rise_index = prev_index
                break

        if rise_index is not None:
            rise_time = rise_index.total_seconds()
            rise_VE_VO2 = VE_VO2[rise_index]
            rise_VE_CO2 = VE_CO2[rise_index]
            #print(f"First rise after nadir found at time {rise_time} sec with VE/VO2 = {rise_VE_VO2} and VE/CO2 = {rise_VE_CO2}")
        else:
            print("No first rise after nadir found.")
        # Find the deflection point of VE/CO2 after nadir_index
        deflection_index = None
        for index in VE_CO2.index[VE_CO2.index > nadir_index]:
            if (index.total_seconds() - nadir_seconds) < 100:
                continue  # Ensure at least 5 seconds have passed
            prev_index = VE_CO2.index[VE_CO2.index < index].max()
            next_index = VE_CO2.index[VE_CO2.index > index].min()
            
            if prev_index is not None and next_index is not None:
                prev_slope = VE_CO2.loc[index] - VE_CO2.loc[prev_index]
                next_slope = VE_CO2.loc[next_index] - VE_CO2.loc[index]
                
                if prev_slope > 0 and next_slope < 0:  # Detect peak or deflection
                    deflection_index = index
                    break

        if deflection_index is not None:
            deflection_time = deflection_index.total_seconds()
            deflection_VE_CO2 = VE_CO2[deflection_index]
            #print(f"Deflection point of VE/CO2 found at time {deflection_time} sec with VE/CO2 = {deflection_VE_CO2}")
        else:
            print("No deflection point of VE/CO2 found after nadir.")
        vt1_speed, vt1_grade = self.get_speed_grade_from_time(nadir_time)
        vt2_speed, vt2_grade = self.get_speed_grade_from_time(deflection_index) if deflection_index is not None else (None, None)

        if plot:
            fig, ax = plt.subplots(figsize=(8, 6))

            ax.plot(time, VE_VO2, color='b', label='VE/VO2', zorder=2)
            ax.plot(time, VE_CO2, color='r', label='VE/CO2', zorder=1)
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Value')
            ax.tick_params(axis='y')

            # Mark the nadir, first rise, and deflection points
            ax.scatter(nadir_seconds, nadir_value, color='black', zorder=3, label='Nadir')
            if rise_index is not None:
                ax.scatter(rise_time, rise_VE_VO2, color='green', zorder=3, label='First Rise')
            if deflection_index is not None:
                ax.scatter(deflection_time, deflection_VE_CO2, color='purple', zorder=3, label='Deflection Point')

            fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
            ax.grid(True)
            plt.title('VE/VO2 and VE/CO2 vs. Time')
            #plt.show()

        return {'vt1_time_VE': nadir_time, 'vt1_speed_VE': vt1_speed, 'vt1_grade_VE': vt1_grade,
                'vt2_time_VE': deflection_index if deflection_index is not None else None, 'vt2_speed_VE': vt2_speed, 'vt2_grade_VE': vt2_grade}

    def get_speed_grade_from_time(self, time):
        #time = pd.to_timedelta(time)

        # Slice everything up to and including 'time'
        sub_df = self.df.loc[:time]
        if sub_df.empty:
            raise ValueError(f"No data available at or before time {time}")

        row = sub_df.iloc[-1]  # Last row before or equal to time
        return row['Speed'], row['Grade']
     
    def vt1_vt2_pet(self, window_size, plot=False):
            # Apply rolling mean
        PetO2 = self.df['PetO2'].rolling(f'{window_size}s', min_periods=1).mean()
        PetCO2 = self.df['PetCO2'].rolling(f'{window_size}s', min_periods=1).mean()
        
        # Convert time index to total seconds
        time = self.df.index.total_seconds()
        # Identify nadir: Minimum PetO2 value
        nadir_index = PetO2.idxmin()
        if pd.isna(nadir_index) or nadir_index not in self.df.index:
            raise ValueError("nadir_index is NaN or not in DataFrame index")

        nadir_time = self.df.index[self.df.index.get_loc(nadir_index)]  # Safer method to get position
        nadir_value = PetO2[nadir_index]

        # Find the first rise after nadir while PetCO2 is constant or increasing
        rise_index = None

        nadir_seconds = nadir_time.total_seconds()  # Convert Timedelta to seconds
        end_seconds = self.df.index[-1].total_seconds()
        for index in PetCO2.index[1:]:
            prev_index = PetCO2.index[PetCO2.index < index].max()
            if PetCO2.loc[index] >= PetCO2.loc[prev_index] and PetO2.loc[index] > nadir_value:
                rise_index = prev_index
                break

        if rise_index is not None:
            rise_time = rise_index.total_seconds()
            rise_PetO2 = PetO2[rise_index]
            rise_PetCO2 = PetCO2[rise_index]
            #print(f"First rise after nadir found at time {rise_time} sec with PetO2 = {rise_PetO2} and PetCO2 = {rise_PetCO2}")
        else:
            print("No first rise after nadir found.")
        # Compute the first derivative (rate of change) --> after aerobic threshold
        mask = PetCO2.index > nadir_index
        PetCO2_post_nadir = PetCO2[mask]
        time_post_nadir = time[mask]

        # Derivatives
        dPetCO2_dt = np.gradient(PetCO2_post_nadir, time_post_nadir)
        d2PetCO2_dt2 = np.gradient(dPetCO2_dt, time_post_nadir)

        # Deflection: point of max acceleration after nadir
        deflection_idx_local = np.argmax(d2PetCO2_dt2)
        deflection_index = PetCO2_post_nadir.index[deflection_idx_local]
        deflection_time = deflection_index
        deflection_value = PetCO2_post_nadir.iloc[deflection_idx_local]
        # Print results
        #print(f"Deflection point found at time {deflection_time} sec with PetCO2 = {deflection_value}")
        # Plot PetO2 and PetCO2
        vt1_speed, vt1_grade = self.get_speed_grade_from_time(nadir_index)
        vt2_speed, vt2_grade = self.get_speed_grade_from_time(deflection_time) if deflection_index is not None else (None, None)
        if plot:
            fig, ax1 = plt.subplots(figsize=(8, 6))

            ax1.plot(time, PetO2, color='b', label='PetO2', zorder=2)
            ax1.set_xlabel('Time (s)')
            ax1.set_ylabel('PetO2 (L/min)', color='b')
            ax1.tick_params(axis='y', labelcolor='b')

            ax2 = ax1.twinx()
            ax2.plot(time, PetCO2, color='r', label='PetCO2', zorder=1)
            ax2.set_ylabel('PetCO2 (L/min)', color='r')
            ax2.tick_params(axis='y', labelcolor='r')

            # Mark the nadir, first rise, and deflection points
            ax1.scatter(nadir_index.total_seconds(), nadir_value, color='black', zorder=3, label='Nadir')
            if rise_index is not None:
                ax1.scatter(rise_time, rise_PetO2, color='green', zorder=3, label='First Rise')
            ax2.scatter(deflection_time.total_seconds(), deflection_value, color='purple', zorder=3, label='Deflection Point')

            fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
            ax1.grid(True)
            plt.title('PetO2 and PetCO2 vs. Time')
            #plt.show()

        return {'vt1_time_pet':nadir_time, 'vt1_speed_pet':vt1_speed, 'vt1_grade_pet':vt1_grade,
                'vt2_time_pet':deflection_time, 'vt2_speed_pet':vt2_speed, 'vt2_grade_pet':vt2_grade}
    
    def get_VO2_peak_time_metrics(self, window_size=30, rq_threshold=1.0):
        """
        Calculates the peak VO2 value over a rolling window (default 30 seconds), and collects associated metrics.

        Returns:
            dict: Contains the peak VO2 value, start/end times, and corresponding physiological metrics.
        """
        # Compute rolling average (requires t to be the datetime index)
        rolling_avg = self.df['VO2'].rolling(f'{window_size}s').mean()

        max_avg = rolling_avg.max()
        end_time = rolling_avg.idxmax()
        start_time = end_time - pd.Timedelta(seconds=30)

        # Handle edge case if there's no peak found
        if pd.isna(max_avg) or pd.isna(start_time) or pd.isna(end_time):
            return {"VO2_peak": None, "start_time": None, "end_time": None}

        # Check if Stage is NaN at end_time
        try:
            if pd.isna(self.df.loc[end_time, 'Stage']):
                grade = self.df.loc[start_time, 'Grade']
                speed = self.df.loc[start_time, 'Speed']
                stage = self.df.loc[start_time, 'Stage']
            else:
                grade = self.df.loc[end_time, 'Grade']
                speed = self.df.loc[end_time, 'Speed']
                stage = self.df.loc[end_time, 'Stage']
        except KeyError:
            # fallback in case the exact time index isn't available (rare)
            nearest_end_idx = self.df.index.get_indexer([end_time], method='nearest')[0]
            end_time = self.df.index[nearest_end_idx]
            start_time = end_time - pd.Timedelta(seconds=30)
            grade = self.df.loc[end_time, 'Grade']
            speed = self.df.loc[end_time, 'Speed']
            stage = self.df.loc[end_time, 'Stage']

        # Slice data between start and end time
        peak_window_df = self.df.loc[start_time:end_time]

        rq_peak = peak_window_df['RQ'].mean()
        hr_peak = peak_window_df['HR'].mean()
        eem_peak = peak_window_df['EEm'].mean()
        fat_pct_peak = peak_window_df['Fat'].mean()
        cho_pct_peak = peak_window_df['CHO'].mean()
        vo2_kg_peak = peak_window_df['VO2/Kg'].mean()

        # First value after VO2 peak
        post_peak = self.df.loc[self.df.index >= end_time]
        pre_peak = self.df.loc[self.df.index < end_time]
        # Lactate: First valid value after peak, or last valid value before if none after
        if not post_peak['La-'].dropna().empty:
            lactate_post = post_peak['La-'].dropna().iloc[0]
        else:
            lactate_post = pre_peak['La-'].dropna().iloc[-1] if not pre_peak['La-'].dropna().empty else None

        # First non-NaN RPE after VO2 peak
        rpe_post = post_peak['Dyspnea'].dropna().iloc[0] if not post_peak['Dyspnea'].dropna().empty else None

        # Boolean for whether RQ exceeds the threshold
        true_vo2max = rq_peak > rq_threshold if not pd.isna(rq_peak) else False

        return {
            "VO2_peak": max_avg,
            "start_time": start_time,
            "end_time": end_time,
            "GradePeak": grade,
            "SpeedPeak": speed,
            "MarkerPeak": stage,
            "RQPeak": rq_peak,
            "HRPeak": hr_peak,
            "EEMPeak": eem_peak,
            "Fat%Peak": fat_pct_peak,
            "CHO%Peak": cho_pct_peak,
            "VO2/kgPeak": vo2_kg_peak,
            "Lactate-VO2Peak": lactate_post,
            "RPE-VO2peak": rpe_post,
            "True VO2max": true_vo2max
        }


In [421]:
#Read in test data into a list of Test objects
path = r'visit_data'
filenames = glob.glob(os.path.join(path, "*.csv"))
test_list = []
for filename in filenames:
    df = pd.read_csv(filename)
    test = Test(df)
    test_list.append(test)


In [436]:
rows = []
for test in test_list:
    try:
        result = test.get_VO2_peak_time_metrics(window_size=30)
    except Exception as e:
        result[f"vo2_max_error"] = str(e)
    result['name'] = test.name
    rows.append(result)
vo2_df = pd.DataFrame(rows).set_index('name')
vo2_df.info()
    

<class 'pandas.core.frame.DataFrame'>
Index: 623 entries, TR000207_20221201 to TR000138_20220414
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype          
---  ------           --------------  -----          
 0   VO2_peak         623 non-null    float64        
 1   start_time       623 non-null    timedelta64[ns]
 2   end_time         623 non-null    timedelta64[ns]
 3   GradePeak        623 non-null    float64        
 4   SpeedPeak        623 non-null    float64        
 5   MarkerPeak       623 non-null    float64        
 6   RQPeak           623 non-null    float64        
 7   HRPeak           623 non-null    float64        
 8   EEMPeak          623 non-null    float64        
 9   Fat%Peak         623 non-null    float64        
 10  CHO%Peak         623 non-null    float64        
 11  VO2/kgPeak       623 non-null    float64        
 12  Lactate-VO2Peak  16 non-null     float64        
 13  RPE-VO2peak      623 non-null    object         
 14  T

In [369]:
funcs_with_args = {
    'lt_abs_vals': {},
    'lt_ref_vals': {},
    'lt_log_semilog': {},
    'vt1_vt2_vo2_vco2': {},
    'vt1_vt2_VE': {'window_size': 30},
    'vt1_vt2_pet': {'window_size': 30},
}

rows = []
for test in test_list:
    combined_result = {}
    for func_name, kwargs in funcs_with_args.items():
        try:
            method = getattr(test, func_name)
            result = method(**kwargs)
            if isinstance(result, dict):
                combined_result.update(result)
            else:
                combined_result[func_name] = result  # fallback if not a dict
        except Exception as e:
            combined_result[f"{func_name}_error"] = str(e)
    combined_result['name'] = test.name
    rows.append(combined_result)

# Create DataFrame
df = pd.DataFrame(rows).set_index('name')
df

/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1242: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2)/(dx1 * (dx1 + dx2))
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1243: RuntimeWarning: divide by zero encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1244: RuntimeWarning: divide by zero encountered in divide
  c = dx1 / (dx2 * (dx1 + dx2))
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1250: RuntimeWarning: invalid value encountered in add
  out[tuple(slice1)] = a * f[tuple(slice2)] + b * f[tuple(slice3)] + c * f[tuple(slice4)]
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1242: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2)/(dx1 * (dx1 + dx2))
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1243: RuntimeWarning: divide by zero encountered in divide
  b 

,LT1_abs,LT2_abs,LT1abs_speed,LT2abs_speed,LT1_ref,LT2_ref,LT1ref_speed,LT2ref_speed,LT1_log,LT1_log_speed,...,vt1_time_pet,vt1_speed_pet,vt1_grade_pet,vt2_time_pet,vt2_speed_pet,vt2_grade_pet,lt_log_semilog_error,vt1_vt2_VE_error,vt1_vt2_vo2_vco2_error,vt1_vt2_pet_error
name,,,,,,,,,,,,,,,,,,,,,
TR000207_20221201,2.5,5.8,8.8,8.8,2.5,3.6,8.8,8.8,1.770679,7.929373,...,0 days 00:07:31,5.7,0.0,0 days 00:34:53,0.0,0.0,NaN,NaN,NaN,NaN
TR000199_20220503,2.2,5.0,2.0,9.9,3.4,5.0,9.3,9.9,2.628375,9.078598,...,0 days 00:02:09,2.0,0.0,0 days 00:22:43,10.5,0.0,NaN,NaN,NaN,NaN
TR000143_20220804,2.0,5.3,2.0,8.8,3.6,3.6,8.8,8.8,2.182407,7.853670,...,0 days 00:12:00,6.3,0.0,0 days 00:29:42,8.8,4.0,NaN,NaN,NaN,NaN
TR000137_20220527,2.0,5.0,2.0,9.9,2.6,5.0,9.3,9.9,2.103836,8.756785,...,0 days 00:08:37,7.4,0.0,0 days 00:35:34,0.0,0.0,NaN,NaN,NaN,NaN
TR000194_20220316,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0 days 00:03:26,0.0,0.0,0 days 00:28:47,11.6,0.0,zero-size array to reduction operation minimum...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TR000122_20220817,2.1,5.3,2.0,8.1,2.7,5.3,6.9,8.1,2.238824,7.500031,...,0 days 00:07:54,5.7,0.0,0 days 00:26:42,2.0,0.0,NaN,NaN,NaN,NaN
TR000223_20220803,2.3,6.5,9.7,10.9,2.3,6.5,9.7,10.9,1.699454,9.227631,...,0 days 00:08:47,7.2,0.0,0 days 00:18:59,9.7,0.0,NaN,NaN,NaN,NaN
TR000242_20220419,2.3,4.7,2.0,5.2,2.5,4.7,4.6,5.2,1.973839,4.269904,...,0 days 00:02:06,0.0,0.0,0 days 00:05:09,0.0,0.0,NaN,NaN,NaN,NaN


In [371]:
error_columns = df.filter(like='error')
value_counts = error_columns.apply(pd.Series.value_counts)
value_counts


,lt_log_semilog_error,vt1_vt2_VE_error,vt1_vt2_vo2_vco2_error,vt1_vt2_pet_error
"The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().",NaN,5.0,NaN,2.0
"bounds should be a sequence containing finite real valued (min, max) pairs for each value in x",NaN,NaN,1.0,NaN
index values must not have NaT,NaN,1.0,NaN,1.0
zero-size array to reduction operation minimum which has no identity,20.0,NaN,NaN,NaN


In [372]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 623 entries, TR000207_20221201 to TR000138_20220414
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype          
---  ------                  --------------  -----          
 0   LT1_abs                 603 non-null    float64        
 1   LT2_abs                 593 non-null    float64        
 2   LT1abs_speed            603 non-null    float64        
 3   LT2abs_speed            593 non-null    float64        
 4   LT1_ref                 603 non-null    float64        
 5   LT2_ref                 595 non-null    float64        
 6   LT1ref_speed            603 non-null    float64        
 7   LT2ref_speed            595 non-null    float64        
 8   LT1_log                 603 non-null    float64        
 9   LT1_log_speed           603 non-null    float64        
 10  LT1_log_r2              602 non-null    float64        
 11  LT2_semilog             603 non-null    float64        
 12  LT2_semilog

In [246]:
path = r'visit_data'
filenames = glob.glob(os.path.join(path, "*.csv"))
schema = {'aerobic_time':'float', 'aerobic_O2_val':'float', 'aerobic_CO2_val':'float', 'anaerobic_time':'float', 'anaerobic_O2_val':'float', 'anaerobic_CO2_val':'float'}
for filename in filenames:
    df = pd.read_csv(filename)
    test = Test(df)
    window_size_df = pd.DataFrame(columns=schema.keys()).astype(schema)
    results = []  # Use a list to store results before concatenating
    for i in range(1, 120):
        try:
            result = test.aerobic_anaerobic_3(i)  # Assuming this returns a dictionary or a Series
            result["window_size"] = i  # Add window size as a column
            results.append(result)  # Store result in list
        except Exception as e:
            print(f"Skipping window size {i} due to error: {e}")

    # Convert results to DataFrame in one step
    '''
    if results:
        window_size_df = pd.concat([window_size_df, pd.DataFrame(results)], ignore_index=True)

    window_size_df
    window_size_df.plot(x='window_size', y=['aerobic_time', 'anaerobic_time'], kind='line')
    plt.xlabel('Window Size')
    plt.ylabel('Time to reach threshold (s)')
    plt.title(f'{test.name}: Aerobic Time and Anaerobic Time vs Window Size')
    plt.legend(['Aerobic Time', 'Anaerobic Time'])
    plt.show()

    
    window_size_df.plot(x='window_size', y=['aerobic_O2_val', 'anaerobic_O2_val'], kind='line')
    plt.xlabel('Window Size')
    plt.title(f'{test.name}: Aerobic and Anaerobic O2 values vs Window Size')
    plt.ylabel('Threshold O2 Value')
    plt.legend(['Aerobic O2', 'Anaerobic O2'])
    plt.show()
    window_size_df.plot(x='window_size', y=['aerobic_CO2_val', 'anaerobic_CO2_val'], kind='line')
    plt.xlabel('Window Size')
    plt.ylabel('Threshold CO2 Value')
    plt.title(f'{test.name}: Aerobic and Anaerobic CO2 values vs Window Size')
    plt.legend(['Aerobic CO2', 'Anaerobic CO2'])
    plt.show()
    '''


/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1242: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2)/(dx1 * (dx1 + dx2))
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1243: RuntimeWarning: divide by zero encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1244: RuntimeWarning: divide by zero encountered in divide
  c = dx1 / (dx2 * (dx1 + dx2))
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1250: RuntimeWarning: invalid value encountered in add
  out[tuple(slice1)] = a * f[tuple(slice2)] + b * f[tuple(slice3)] + c * f[tuple(slice4)]
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1242: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2)/(dx1 * (dx1 + dx2))
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/function_base.py:1243: RuntimeWarning: divide by zero encountered in divide
  b 

In [149]:
path = r'visit_data'
filenames = glob.glob(os.path.join(path, "*.csv"))

folder_name = "vo2_vco2_graphs"

for filename in filenames:
    df = pd.read_csv(filename)
    test = Test(df)
    test_name = test.name  # Assuming each test has a unique name
    try:
        fig, vo2_threshold, vco2_threshold = test.aerobic_1()  # Correct unpacking
        fig.savefig(f'{folder_name}/{test_name}_aerobic_1.png')
        plt.close(fig)  # Free memory
        
        print(f"{test_name}: Aerobic 1 threshold: {vo2_threshold:.2f} L/min, {vco2_threshold:.2f} L/min")
    except Exception as e:
        print(f"Error processing {test_name}: {e}")

/var/folders/00/25nsh44n6l12wt18fkf86ns80000gn/T/ipykernel_18537/1844921518.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df['Stage'].fillna(0, inplace=True)  # Ensure non-exercise rows have Stage 0


TR000105_20220207: Aerobic 1 threshold: -1382.24 L/min, -1517.95 L/min


/var/folders/00/25nsh44n6l12wt18fkf86ns80000gn/T/ipykernel_18537/1844921518.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df['Stage'].fillna(0, inplace=True)  # Ensure non-exercise rows have Stage 0


TR000106_20220420: Aerobic 1 threshold: -339.70 L/min, -568.94 L/min


/var/folders/00/25nsh44n6l12wt18fkf86ns80000gn/T/ipykernel_18537/1844921518.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df['Stage'].fillna(0, inplace=True)  # Ensure non-exercise rows have Stage 0


...

TR000101_20220331: Aerobic 1 threshold: -679.88 L/min, -850.67 L/min


/var/folders/00/25nsh44n6l12wt18fkf86ns80000gn/T/ipykernel_18537/1844921518.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df['Stage'].fillna(0, inplace=True)  # Ensure non-exercise rows have Stage 0


TR000107_20220805: Aerobic 1 threshold: -2307.16 L/min, -2439.53 L/min


/var/folders/00/25nsh44n6l12wt18fkf86ns80000gn/T/ipykernel_18537/1844921518.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df['Stage'].fillna(0, inplace=True)  # Ensure non-exercise rows have Stage 0


TR000109_20220518: Aerobic 1 threshold: -628.40 L/min, -741.27 L/min


In [240]:
import pandas as pd
import glob
import os

path = r'visit_data'
filenames = glob.glob(os.path.join(path, "*.csv"))

results = []
folder_name = "lactate_graphs"

for filename in filenames:
    df = pd.read_csv(filename)
    test = Test(df)
    test_name = test.name  # Assuming each test has a unique name
    test_results = test.lt_ref_vals()
    test_results.update(test.lt_abs_vals())
    if test.lt_log_semilog():
        test_results.update(test.lt_log_semilog())
    test_results['Test Name'] = test_name
    
    # Append results to the list
    results.append(test_results)

# Convert results into a DataFrame
results_df = pd.DataFrame(results).set_index('Test Name')

# Display the DataFrame (optional)
results_df

,LT1_ref,LT2_ref,LT1ref_speed,LT2ref_speed,LT1_abs,LT2_abs,LT1abs_speed,LT2abs_speed,LT1_log,LT1_log_speed,LT1_log_r2,LT2_semilog,LT2_semilog_speed,LT2_semi_log_r2
Test Name,,,,,,,,,,,,,,
TR000105_20220207,3.6,3.6,4.8,4.8,2.1,4.7,2.0,6.0,5.574446,7.015555,0.855457,6.863055,6.136242,0.914631
TR000106_20220420,2.4,4.3,7.1,8.3,2.4,4.3,7.1,8.3,1.906243,6.594936,0.949458,6.593568,1.910062,0.952030
TR000102_20220204,4.2,4.2,6.8,6.8,3.2,4.2,2.0,6.8,3.701032,8.985182,0.315441,8.967361,3.806786,0.329163
TR000100_20220623,2.5,5.0,8.6,9.8,2.9,5.0,2.0,9.8,2.003482,8.492251,0.567424,8.407638,1.946956,0.584825
TR000104_20220627,3.2,10.4,8.8,8.8,2.6,10.4,2.0,8.8,2.232814,8.687540,0.501426,8.622199,2.191050,0.533642
TR000100_20220331,2.1,3.5,9.2,9.8,2.1,4.1,2.0,2.0,1.592061,8.415150,0.750100,8.391522,1.559386,0.779775
TR000109_20220209,4.1,4.1,6.9,6.9,3.1,4.1,2.0,6.9,4.487763,7.909333,0.544670,7.936506,4.677765,0.621858
TR000107_20220519,3.5,3.5,8.0,8.0,2.1,7.5,2.0,10.5,2.168712,10.029869,0.360399,9.978255,2.122445,0.358410
TR000106_20220208,3.1,5.8,5.8,6.5,2.6,5.8,2.0,6.5,5.982798,8.063484,0.550467,8.164476,6.641185,0.561781


In [116]:
results_df['LT1/LT2_Ref_same'] = results_df['LT1_ref'] == results_df['LT2_ref']
results_df['LT1/LT2_Abs_same'] = results_df['LT1_abs'] == results_df['LT2_abs']
results_df['LT1_Ref_Abs_same'] = results_df['LT1_ref'] == results_df['LT1_abs']
results_df['LT2_Ref_Abs_same'] = results_df['LT2_ref'] == results_df['LT2_abs']
results_df['LT1log_speed>LT2log_speed'] = results_df['LT1_log_speed'] > results_df['LT2_semilog_speed']
results_df['LT1_log>LT2_semilog'] = results_df['LT1_log'] > results_df['LT2_semilog']
results_df = results_df.sort_index()
results_df.sort_values(by='LT1_log_r2', ascending=False)


,LT1_ref,LT2_ref,LT1ref_speed,LT2ref_speed,LT1_abs,LT2_abs,LT1abs_speed,LT2abs_speed,LT1_log,LT1_log_speed,LT1_log_r2,LT2_semilog,LT2_semilog_speed,LT2_semi_log_r2,LT1/LT2_Ref_same,LT1/LT2_Abs_same,LT1_Ref_Abs_same,LT2_Ref_Abs_same,LT1log_speed>LT2log_speed,LT1_log>LT2_semilog
Test Name,,,,,,,,,,,,,,,,,,,,
TR000108_20220209,3.7,3.7,5.4,5.4,2.1,5.5,2.0,6.6,3.915427,6.006571,0.999937,6.047295,4.007126,0.997640,True,False,False,False,True,False
TR000101_20220331,3.4,4.8,6.5,8.3,2.6,4.8,2.0,8.3,3.569491,7.682926,0.998524,7.712744,3.622065,0.999232,False,False,False,True,True,False
TR000110_20220428,2.1,3.3,6.3,7.5,2.1,5.9,6.3,8.1,2.127867,6.922036,0.994310,6.906890,2.144034,0.989008,False,False,True,False,True,False
TR000103_20220204,3.6,3.6,8.0,8.0,2.1,4.2,2.0,8.6,3.911724,8.378838,0.982948,8.381042,3.928585,0.989746,True,False,False,False,True,False
TR000103_20220401,5.9,5.9,9.3,9.3,3.5,5.9,2.0,9.3,2.213647,8.634403,0.973741,8.627891,2.188238,0.982329,True,False,False,True,True,False
TR000106_20220420,2.4,4.3,7.1,8.3,2.4,4.3,7.1,8.3,1.906243,6.594936,0.949458,6.593568,1.910062,0.952030,False,False,True,True,True,False
TR000109_20220824,4.3,4.3,8.8,8.8,2.2,4.3,6.3,8.8,2.382520,8.137912,0.948005,8.159454,2.425536,0.931522,True,False,False,True,True,False
TR000109_20220518,2.8,4.4,6.3,7.5,2.2,4.4,2.0,7.5,3.060430,6.692052,0.946224,6.631433,3.058710,0.956863,False,False,False,True,True,False
TR000110_20220901,1.8,6.8,6.3,8.1,2.3,6.8,6.9,8.1,2.073628,6.853761,0.926557,6.834631,2.096206,0.940022,False,False,False,True,True,False


In [130]:
results_df['LT1_log - LT1 ref'] = results_df['LT1_log'] - results_df['LT1_ref']
results_df['LT2_semilog - LT2 ref'] = results_df['LT2_semilog'] - results_df['LT2_ref']
results_df['LT1_log - LT1_abs'] = results_df['LT1_log'] - results_df['LT1_abs']
results_df['LT2_semilog - LT2_abs'] = results_df['LT2_semilog'] - results_df['LT2_abs']
results_df = results_df[results_df['LT2_semilog'] < 20]
print(abs(results_df['LT1_log - LT1_abs']).describe())
print(abs(results_df['LT2_semilog - LT2_abs']).describe())
#print(abs(results_df['LT1_log - LT1 ref']).describe())
#print(abs(results_df['LT2_semilog - LT2 ref']).describe())

count    30.000000
mean      1.024404
std       0.959331
min       0.027867
25%       0.266281
50%       0.873730
75%       1.401279
max       3.474446
Name: LT1_log - LT1_abs, dtype: float64
count    30.000000
mean      2.708238
std       1.620058
min       0.034631
25%       1.728784
50%       2.329022
75%       4.100645
max       5.648262
Name: LT2_semilog - LT2_abs, dtype: float64


/var/folders/00/25nsh44n6l12wt18fkf86ns80000gn/T/ipykernel_57705/4271827488.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_df['LT1_log - LT1 ref'] = results_df['LT1_log'] - results_df['LT1_ref']
/var/folders/00/25nsh44n6l12wt18fkf86ns80000gn/T/ipykernel_57705/4271827488.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_df['LT2_semilog - LT2 ref'] = results_df['LT2_semilog'] - results_df['LT2_ref']
/var/folders/00/25nsh44n6l12wt18fkf86ns80000gn/T/ipykernel_57705/4271827488.py:3: Settin

In [129]:
results_df['LT2_semilog'].sort_values(ascending=False)

Test Name
TR000107_20220519    9.978255
TR000107_20220805    9.819362
TR000103_20220706    9.648262
TR000102_20220401    9.556322
TR000107_20220208    9.536177
TR000102_20220204    8.967361
TR000103_20220401    8.627891
TR000104_20220627    8.622199
TR000102_20220630    8.614629
TR000100_20220623    8.407638
TR000100_20220331    8.391522
TR000103_20220204    8.381042
TR000104_20220314    8.270761
TR000106_20220208    8.164476
TR000109_20220824    8.159454
TR000109_20220209    7.936506
TR000101_20220331    7.712744
TR000106_20220823    7.567625
TR000110_20220210    7.440470
TR000101_20220630    7.071249
TR000110_20220428    6.906890
TR000105_20220920    6.895285
TR000105_20220207    6.863055
TR000110_20220901    6.834631
TR000109_20220518    6.631433
TR000106_20220420    6.593568
TR000105_20220517    6.322675
TR000108_20220209    6.047295
TR000104_20220207    4.826963
TR000101_20220203    0.668274
Name: LT2_semilog, dtype: float64